In [1]:
# ─────────────────────────────────────────────
# CELL 1 — Check GPU
# ─────────────────────────────────────────────
import torch
print('PyTorch version:', torch.__version__)
print('GPU available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name       :', torch.cuda.get_device_name(0))
    print('✅ GPU ready!')
else:
    print('⚠️  GPU not found!')
    print('   Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save')
    print('   Then re-run all cells.')

PyTorch version: 2.11.0+cpu
GPU available  : False
⚠️  GPU not found!
   Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save
   Then re-run all cells.


In [2]:
# ─────────────────────────────────────────────
# CELL 2 — Install dependencies
# ─────────────────────────────────────────────
!pip install kaggle -q
print('✅ Dependencies ready')

✅ Dependencies ready


In [3]:
# ─────────────────────────────────────────────
# CELL 3 — Download PlantVillage Dataset from Kaggle
# FIX: Added error handling + permission fix
# ─────────────────────────────────────────────
import os, json

# ✏️ YOUR KAGGLE CREDENTIALS
KAGGLE_USERNAME = 'yashrakholiya007'          # <-- your Kaggle username
KAGGLE_KEY      = 'KGAT_71b91a72bc8c2f42924f281353f51284'  # <-- your Kaggle API key

# Setup kaggle credentials
os.makedirs('/root/.kaggle', exist_ok=True)
kaggle_json = {'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_json, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# FIX: Also set env variable (some Colab versions need this)
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY']      = KAGGLE_KEY

print('Downloading PlantVillage dataset (~2.5 GB)...')
print('This may take 5-10 minutes...')

result = os.system('kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/ --unzip -q')

if result == 0:
    print('✅ Dataset downloaded successfully!')
else:
    print('❌ Download failed! Check your Kaggle credentials.')
    print('   Go to kaggle.com → Account → API → Create New Token')
    print('   Copy username and key, paste above and re-run.')

This may take 5-10 minutes...
✅ Dataset downloaded successfully!


In [4]:
# ─────────────────────────────────────────────
# CELL 4 — Find dataset path
# FIX: More robust path detection
# ─────────────────────────────────────────────
import os

BASE = None

# Method 1: Walk and find folder with Apple___Apple_scab
for root, dirs, files in os.walk('/content'):
    if 'Apple___Apple_scab' in dirs:
        BASE = root
        break

# Method 2: Try known paths
if BASE is None:
    candidate_paths = [
        '/content/plantvillage dataset/color',
        '/content/plantvillage-dataset/color',
        '/content/plantvillage_dataset/color',
        '/content/color',
        '/content/PlantVillage',
        '/content/plantvillage dataset',
    ]
    for path in candidate_paths:
        if os.path.exists(path) and os.path.isdir(path):
            items = os.listdir(path)
            # Check if it has disease class folders
            if any('Apple' in item for item in items):
                BASE = path
                break

# Method 3: Show all /content folders to debug
if BASE is None:
    print('❌ Dataset not found! Showing /content structure:')
    for item in os.listdir('/content'):
        full = os.path.join('/content', item)
        if os.path.isdir(full):
            sub = os.listdir(full)[:5]
            print(f'  📁 {item}/ → {sub}')
    raise FileNotFoundError('Dataset not found. Re-run Cell 3.')

print(f'✅ Dataset found at: {BASE}')
classes = sorted([d for d in os.listdir(BASE) if os.path.isdir(os.path.join(BASE, d))])
print(f'   Total classes: {len(classes)}')
total_images = 0
for i, c in enumerate(classes):
    count = len(os.listdir(os.path.join(BASE, c)))
    total_images += count
    print(f'  {i:2d}. {c} — {count} images')
print(f'\n   Total images: {total_images:,}')

✅ Dataset found at: /content\plantvillage dataset\color
   Total classes: 38
   0. Apple___Apple_scab — 630 images
   1. Apple___Black_rot — 621 images
   2. Apple___Cedar_apple_rust — 275 images
   3. Apple___healthy — 1645 images
   4. Blueberry___healthy — 1502 images
   5. Cherry_(including_sour)___Powdery_mildew — 1052 images
   6. Cherry_(including_sour)___healthy — 854 images
   7. Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot — 513 images
   8. Corn_(maize)___Common_rust_ — 1192 images
   9. Corn_(maize)___Northern_Leaf_Blight — 985 images
  10. Corn_(maize)___healthy — 1162 images
  11. Grape___Black_rot — 1180 images
  12. Grape___Esca_(Black_Measles) — 1383 images
  13. Grape___Leaf_blight_(Isariopsis_Leaf_Spot) — 1076 images
  14. Grape___healthy — 423 images
  15. Orange___Haunglongbing_(Citrus_greening) — 5507 images
  16. Peach___Bacterial_spot — 2297 images
  17. Peach___healthy — 360 images
  18. Pepper,_bell___Bacterial_spot — 997 images
  19. Pepper,_bell___heal

In [5]:
# ─────────────────────────────────────────────
# CELL 5 — Imports & Config
# ─────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import time, copy, os

# ── Config ──
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 15
LR          = 1e-3
# FIX: num_workers=0 on Colab avoids multiprocessing errors
NUM_WORKERS = 2 if torch.cuda.is_available() else 0

print(f'Device      : {DEVICE}')
print(f'Batch size  : {BATCH_SIZE}')
print(f'Epochs      : {EPOCHS}')
print(f'LR          : {LR}')
print(f'Num workers : {NUM_WORKERS}')

Device      : cpu
Batch size  : 32
Epochs      : 15
LR          : 0.001
Num workers : 0


In [6]:
# ─────────────────────────────────────────────
# CELL 6 — Data Transforms
# ─────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print('✅ Transforms ready')

✅ Transforms ready


In [7]:
# ─────────────────────────────────────────────
# CELL 7 — Load Dataset & Split
# FIX: TransformSubset now correctly handles PIL images
# ─────────────────────────────────────────────

# FIX: TransformSubset — handles both PIL and tensor correctly
class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        # FIX: img from ImageFolder(transform=None) is PIL Image
        # Convert to RGB to handle grayscale/RGBA images
        if not isinstance(img, Image.Image):
            # Already a tensor (should not happen, but safety check)
            return img, label
        img = img.convert('RGB')  # FIX: Ensure RGB (handles RGBA, L mode)
        return self.transform(img), label

# Load full dataset WITHOUT transform (returns PIL images)
full_pil    = ImageFolder(root=BASE, transform=None)
CLASS_NAMES = full_pil.classes
NUM_CLASSES = len(CLASS_NAMES)

total  = len(full_pil)
val_sz = int(total * 0.15)
trn_sz = total - val_sz

# FIX: Use same seed so train/val split is consistent
generator = torch.Generator().manual_seed(42)
train_raw, val_raw = random_split(full_pil, [trn_sz, val_sz], generator=generator)

train_dataset = TransformSubset(train_raw, train_transform)
val_dataset   = TransformSubset(val_raw,   val_transform)

# FIX: persistent_workers=False when num_workers=0
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0)
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0)
)

print(f'Total images  : {total:,}')
print(f'Train samples : {trn_sz:,}')
print(f'Val samples   : {val_sz:,}')
print(f'Classes       : {NUM_CLASSES}')
print('✅ DataLoaders ready!')
print('\nAll classes:')
for i, c in enumerate(CLASS_NAMES):
    print(f'  {i:2d}. {c}')

Total images  : 54,305
Train samples : 46,160
Val samples   : 8,145
Classes       : 38
✅ DataLoaders ready!

All classes:
   0. Apple___Apple_scab
   1. Apple___Black_rot
   2. Apple___Cedar_apple_rust
   3. Apple___healthy
   4. Blueberry___healthy
   5. Cherry_(including_sour)___Powdery_mildew
   6. Cherry_(including_sour)___healthy
   7. Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
   8. Corn_(maize)___Common_rust_
   9. Corn_(maize)___Northern_Leaf_Blight
  10. Corn_(maize)___healthy
  11. Grape___Black_rot
  12. Grape___Esca_(Black_Measles)
  13. Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
  14. Grape___healthy
  15. Orange___Haunglongbing_(Citrus_greening)
  16. Peach___Bacterial_spot
  17. Peach___healthy
  18. Pepper,_bell___Bacterial_spot
  19. Pepper,_bell___healthy
  20. Potato___Early_blight
  21. Potato___Late_blight
  22. Potato___healthy
  23. Raspberry___healthy
  24. Soybean___healthy
  25. Squash___Powdery_mildew
  26. Strawberry___Leaf_scorch
  27. Strawberry___

In [8]:
# ─────────────────────────────────────────────
# CELL 8 — Build EfficientNet-B0 Model
# FIX: Use pretrained weights for better accuracy
# ─────────────────────────────────────────────

# FIX: Use pretrained ImageNet weights → fine-tune → much better accuracy
model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

# Replace classifier head
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(DEVICE)

# Loss with label smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# FIX: Different LR for backbone vs classifier head
backbone_params    = [p for n, p in model.named_parameters() if 'classifier' not in n]
classifier_params  = [p for n, p in model.named_parameters() if 'classifier' in n]
optimizer = optim.AdamW([
    {'params': backbone_params,   'lr': LR * 0.1},  # slower for pretrained backbone
    {'params': classifier_params, 'lr': LR},         # faster for new head
], weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Mixed precision scaler (only for GPU)
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

total_params = sum(p.numel() for p in model.parameters())
print(f'Model          : EfficientNet-B0 (pretrained ImageNet)')
print(f'Total params   : {total_params:,}')
print(f'Output classes : {NUM_CLASSES}')
print(f'Device         : {DEVICE}')
print(f'Mixed precision: {scaler is not None}')

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\Yash Rakholiya/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:05<00:00, 3.90MB/s]


Model          : EfficientNet-B0 (pretrained ImageNet)
Total params   : 4,056,226
Output classes : 38
Device         : cpu
Mixed precision: False


In [2]:
# ─────────────────────────────────────────────
# CELL 9 — Training Loop
# FIX: Added mixed precision, better error handling
# ─────────────────────────────────────────────
history    = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_acc   = 0.0
best_model = None

print('🚀 Training started...')
print('='*65)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── TRAIN ──
    model.train()
    train_loss, train_correct = 0.0, 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()

        # FIX: Mixed precision training (2x faster on GPU)
        if scaler is not None:
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss    = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        train_loss    += loss.item() * images.size(0)
        _, predicted   = outputs.max(1)
        train_correct += predicted.eq(labels).sum().item()

        if (batch_idx + 1) % 50 == 0:
            print(f'  Epoch {epoch}/{EPOCHS} | Batch {batch_idx+1}/{len(train_loader)}', end='\r')

    # ── VALIDATE ──
    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            # FIX: autocast for val too
            if scaler is not None:
                with torch.cuda.amp.autocast():
                    outputs = model(images)
                    loss    = criterion(outputs, labels)
            else:
                outputs = model(images)
                loss    = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()

    scheduler.step()

    t_loss = train_loss / trn_sz
    v_loss = val_loss   / val_sz
    t_acc  = train_correct / trn_sz * 100
    v_acc  = val_correct   / val_sz * 100
    elapsed = time.time() - t0

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    star = '⭐' if v_acc > best_acc else '  '
    if v_acc > best_acc:
        best_acc   = v_acc
        best_model = copy.deepcopy(model.state_dict())

    print(f'Epoch {epoch:2d}/{EPOCHS} {star} | '
          f'Train: {t_acc:.2f}% ({t_loss:.4f}) | '
          f'Val: {v_acc:.2f}% ({v_loss:.4f}) | '
          f'{elapsed:.0f}s')

print('='*65)
print(f'✅ Training complete! Best Val Accuracy: {best_acc:.2f}%')

🚀 Training started...


NameError: name 'EPOCHS' is not defined

In [3]:
# ─────────────────────────────────────────────
# CELL 10 — Save Model
# FIX: Save full checkpoint (compatible with app.py load_model_cached)
# ─────────────────────────────────────────────
import os
SAVE_PATH = '/content/plantvillage_efficientnet_b0.pth'

# FIX: Save full checkpoint dict — app.py expects this format
torch.save({
    'model_state_dict': best_model,
    'class_names':      CLASS_NAMES,
    'model_name':       'efficientnet_b0',
    'img_size':         IMG_SIZE,
    'num_classes':      NUM_CLASSES,
    'best_val_acc':     round(best_acc, 4),
    'epochs_trained':   EPOCHS,
}, SAVE_PATH)

size_mb = os.path.getsize(SAVE_PATH) / (1024 * 1024)
print(f'✅ Model saved: {SAVE_PATH}')
print(f'   File size  : {size_mb:.1f} MB')
print(f'   Classes    : {NUM_CLASSES}')
print(f'   Best Acc   : {best_acc:.2f}%')
print()
print('Class names (for reference):')
for i, c in enumerate(CLASS_NAMES):
    print(f"  {i:2d}. '{c}'")

NameError: name 'torch' is not defined

In [ ]:
# ─────────────────────────────────────────────
# CELL 11 — Training Curves
# ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
ax1.plot(history['val_loss'],   'r-o', label='Val Loss',   linewidth=2)
ax1.set_title('Loss Curve', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], 'b-o', label='Train Acc', linewidth=2)
ax2.plot(history['val_acc'],   'r-o', label='Val Acc',   linewidth=2)
ax2.axhline(y=best_acc, color='green', linestyle='--', label=f'Best: {best_acc:.1f}%')
ax2.set_title('Accuracy Curve', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('🌿 PlantAI — EfficientNet-B0 Training Results', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved')

In [ ]:
# ─────────────────────────────────────────────
# CELL 12 — Download model
# ─────────────────────────────────────────────
from google.colab import files
print('Downloading model file...')
files.download('/content/plantvillage_efficientnet_b0.pth')
files.download('/content/training_curves.png')
print('✅ Download started!')
print()
print('NEXT STEPS:')
print('  1. plantvillage_efficientnet_b0.pth → PlantAI project folder ma muko')
print('  2. app.py ma load_model_cached() already compatible che!')
print('  3. streamlit run app.py')

In [ ]:
# ─────────────────────────────────────────────
# CELL 13 — Test prediction
# FIX: Added error handling
# ─────────────────────────────────────────────
import random
from PIL import Image

model.load_state_dict(best_model)
model.eval()

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

all_images = []
for cls in CLASS_NAMES:
    cls_path = os.path.join(BASE, cls)
    if not os.path.isdir(cls_path): continue
    imgs = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if imgs:
        all_images.append((os.path.join(cls_path, random.choice(imgs)), cls))

print('🔍 Testing 5 random images:')
print('-' * 55)
correct = 0
test_samples = random.sample(all_images, min(5, len(all_images)))
for img_path, true_label in test_samples:
    try:
        img    = Image.open(img_path).convert('RGB')  # FIX: convert RGB
        tensor = test_transform(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            out        = model(tensor)
            prob       = torch.softmax(out, dim=1)[0]
            pred_idx   = prob.argmax().item()
            pred_conf  = prob[pred_idx].item() * 100
            pred_label = CLASS_NAMES[pred_idx]
        is_correct = (pred_label == true_label)
        if is_correct: correct += 1
        icon = '✅' if is_correct else '❌'
        print(f'{icon} True: {true_label}')
        print(f'   Pred: {pred_label} ({pred_conf:.1f}%)')
        print()
    except Exception as e:
        print(f'⚠️ Error on {img_path}: {e}')

print(f'Quick test: {correct}/{len(test_samples)} correct')

## ✅ Training Complete!

### FIXED Issues in this notebook:

| # | Problem | Fix |
|---|---------|-----|
| 1 | GPU not detected | Runtime → T4 GPU set karo |
| 2 | Dataset path not found | Auto-detection improved |
| 3 | TransformSubset PIL error | `.convert('RGB')` added |
| 4 | DataLoader worker error | `num_workers=0` when no GPU |
| 5 | Model save format mismatch | Full checkpoint dict saved |
| 6 | Slow training | Mixed precision AMP added |
| 7 | Low accuracy | Pretrained ImageNet weights used |

### app.py ma model load:
```python
# load_model_cached() ma path change karo:
st.session_state.model = load_model_cached('plantvillage_efficientnet_b0.pth')
```

### Project folder:
```
PlantAI/
├── app.py
├── plantvillage_efficientnet_b0.pth  ← downloaded file
